In [1]:
import requests
import bs4 as beautifulsoup

In [2]:
import requests
response = requests.get('https://www.sbtjapan.com/used-cars/search')

In [3]:
soup = beautifulsoup.BeautifulSoup(response.text, 'html.parser')

cars = soup.find_all('li', class_='search-result__item')

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

session = requests.Session()

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
}

all_cars = []

for page in range(1, 50):
    try:
        url = f"https://www.sbtjapan.com/used-cars/search?page={page}"

        response = session.get(
            url,
            headers=headers,
            timeout=30
        )

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        cars = soup.find_all("li", class_="search-result__item")

        print(f"Page {page}: {len(cars)} cars")

        for car in cars:

            try:
                make_tag = car.find("input", {"name": "make"})
                model_tag = car.find("input", {"name": "model"})
                year_tag = car.find("input", {"name": "year"})
                body_tag = car.find("input", {"name": "body_type"})
                make = make_tag.get("value") if make_tag else None
                model = model_tag.get("value") if model_tag else None
                year = year_tag.get("value") if year_tag else None
                body_type = body_tag.get("value") if body_tag else None

                # Vehicle title
                title_tag = car.find("span", class_="card-product__product heading-h2")
                vehicle_name = (title_tag.get_text(strip=True)if title_tag else None)

                # Stock ID
                stock_tag = car.find("span", class_="card-product__stock-value")
                stock_id = (stock_tag.get_text(strip=True)if stock_tag else None)

                # Prices
                price_tags = car.find_all("span", class_="card-product__price")
                price = (price_tags[0].get_text(strip=True)if len(price_tags) > 0 else None)
                total_price = (price_tags[1].get_text(strip=True)if len(price_tags) > 1 else None)

                # Defaults
                mileage = None
                engine_size = None
                fuel_type = None
                transmission = None
                model_code = None
                drive_type = None
                seats = None
                color = None

                # Status fields
                statuses = car.find_all("span", class_=lambda c: c and "card-product__status" in c)

                for status in statuses:
                    classes = status.get("class", [])
                    text = status.get_text(strip=True)

                    if "-mileage" in classes:
                        mileage = text
                    elif "-engine-capacity" in classes:
                        engine_size = text
                    elif "-fuel-type" in classes:
                        fuel_type = text
                    elif "-transmission" in classes:
                        transmission = text
                    elif "-model-code" in classes:
                        model_code = text
                    elif "-drive-type" in classes:
                        drive_type = text
                    elif "-seats" in classes:
                        seats = text
                    elif "-body-color" in classes:
                        color = text

                # Binary 4WD feature
                is_4wd = (1 if drive_type and drive_type.upper() == "4WD"else 0)

                # Vehicle URL
                link_tag = car.find("a", href=True)

                vehicle_url = (
                    "https://www.sbtjapan.com" + link_tag["href"]
                    if link_tag
                    else None
                )

                all_cars.append({
                    "Stock_ID": stock_id,
                    "Vehicle_Name": vehicle_name,
                    "Price": price,
                    "Total_Price": total_price,
                    "Make": make,
                    "Model": model,
                    "Year": year,
                    "Body_Type": body_type,
                    "Mileage": mileage,
                    "Engine_Size": engine_size,
                    "Fuel_Type": fuel_type,
                    "Transmission": transmission,
                    "Drive_Type": drive_type,
                    "Is_4WD": is_4wd,
                    "Seats": seats,
                    "Color": color,
                    "Model_Code": model_code,
                    "Vehicle_URL": vehicle_url,
                    "Source_Platform": "SBT Japan"
                })

            except Exception as e:
                print(f"Vehicle error: {e}")

        time.sleep(2)

    except Exception as e:
        print(f"Page {page} failed: {e}")

df = pd.DataFrame(all_cars)

print("\nPreview:")
print(df.head())
print(f"\nTotal Cars Scraped: {len(df)}")
df.to_csv("sbt_japan_cars.csv", index=False)
print("\nCSV saved as: sbt_japan_cars.csv")

Page 1: 50 cars
Page 2: 50 cars
Page 3: 50 cars
Page 4: 50 cars
Page 5: 50 cars
Page 6: 50 cars
Page 7: 50 cars
Page 8: 50 cars
Page 9: 50 cars
Page 10: 50 cars
Page 11: 50 cars
Page 12: 50 cars
Page 13: 50 cars
Page 14: 50 cars
Page 15: 50 cars
Page 16: 50 cars
Page 17: 50 cars
Page 18: 50 cars
Page 19: 50 cars
Page 20: 50 cars
Page 21: 50 cars
Page 22: 50 cars
Page 23: 50 cars
Page 24: 50 cars
Page 25: 50 cars
Page 26: 50 cars
Page 27: 50 cars
Page 28: 50 cars
Page 29: 50 cars
Page 30: 50 cars
Page 31: 50 cars
Page 32: 50 cars
Page 33: 50 cars
Page 34: 50 cars
Page 35: 50 cars
Page 36: 50 cars
Page 37: 50 cars
Page 38: 50 cars
Page 39: 50 cars
Page 40: 50 cars
Page 41: 50 cars
Page 42: 50 cars
Page 43: 50 cars
Page 44: 50 cars
Page 45: 50 cars
Page 46: 50 cars
Page 47: 50 cars
Page 48: 50 cars
Page 49: 50 cars

Preview:
  Stock_ID                                       Vehicle_Name  Price  \
0   AK8309             2019/5 TOYOTA COROLLA FIELDER HYBRID G  6,700   
1   AK9625            